# Phân tích thành phần Conv-SAITS

| ID | Mô hình | Thay đổi |
|---|---|---|
| A0 | SAITS | Baseline attention |
| A1 | Conv-SAITS Full | Multi-scale kernels 3, 7, 15 |
| A2 | w/o Depth Conv | Bỏ convolution theo depth, giữ BatchNorm |
| A3 | w/o Channel Mixing | Bỏ nhánh residual trộn hidden channels |
| A4 | w/o Variable Mixing | Bỏ nhánh residual trộn các log trong encoder |
| A5 | Single-scale | Một kernel K=15 |

In [ ]:
!git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
%cd /kaggle/working/well-log-imputation
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from dataclasses import asdict
import gc
import json
import logging
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.metrics import compute_imputation_metrics
from src.experiments.data import load_and_check_data
from src.experiments.utils import save_json, set_seed
from src.models._training import synchronized_time
from src.models.saits import SAITS, SAITSConfig
from src.models.conv_saits import ConvSAITS, ConvSAITSConfig

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)


## 2. Dữ liệu và cấu hình chung

Kết quả lưu trực tiếp trong `/kaggle/working/ablation-results/`, theo `<variant>/seed_<seed>/`. Các bảng tổng hợp dùng tên cố định và chỉ lấy variants/seeds trong cấu hình hiện tại. Chạy lại ghi đè file cùng tên, không tạo thư mục theo ngày giờ.


In [ ]:
DATA_ROOT = Path('/kaggle/input/datasets/minhduc0912/geolink-dataset')
OUTPUT_ROOT = Path('/kaggle/working/ablation-results')
SEEDS = (912, 219)
EPOCHS = 200
PATIENCE = 30
SAVE_PREDICTIONS = True
VARIANTS = ("A0", "A1", "A2", "A3", "A4", "A5")
LABELS = {
    "A0": "SAITS", "A1": "Conv-SAITS Full", "A2": "w/o Depth Conv",
    "A3": "w/o Channel Mixing", "A4": "w/o Variable Mixing", "A5": "Single-scale",
}
DEVICE = "cuda"

In [ ]:
preprocessing, data, metadata, data_hashes = load_and_check_data(DATA_ROOT, True)
SCENARIOS = tuple(preprocessing["missing_scenarios"])
LOGS = tuple(preprocessing["log_columns"])
train_data = data["train"]
validation_data = {
    **data["val"][SCENARIOS[0]],
    "validation_scenarios": data["val"],
}

COMMON = dict(
    seq_len=preprocessing["segment_length"], n_features=len(LOGS),
    d_model=256, d_inner=128, n_layers=2, n_heads=4,
    learning_rate=5e-4, batch_size=32, epochs=EPOCHS, patience=PATIENCE,
    dropout=0.1, attn_dropout=0.1, diagonal_attention_mask=True,
    ort_weight=1.0, mit_weight=1.0, mit_reduction="segment",
    independent_shuffle=True, min_delta=0.0, device=DEVICE,
)
CONV = dict(
    encoder_channels=8, kernel_size=15, conv_expansion=2,
    multi_scale_conv=True, multi_scale_kernels=(3, 7, 15),
    masking_strategy="mixed", masking_rate=0.2,
    encoder_norm="batch", temporal_residual=False,
    gap_aware_gate=False, decoder="shared", depth_encoding=False,
    cross_log_attention=False, mit_mse_weight=0.0, mit_relative_weight=0.0,
    gradient_clip=None, validation_objective="rmse",
)

RUN_DIR = OUTPUT_ROOT
RUN_DIR.mkdir(parents=True, exist_ok=True)
save_json(RUN_DIR / "protocol.json", {
    "save_predictions": SAVE_PREDICTIONS,
    "python": platform.python_version(), "torch": torch.__version__,
    "split_sizes": {"train": len(train_data["X"]),
                    "val": len(data["val"][SCENARIOS[0]]["X"]),
                    "test": len(data["test"][SCENARIOS[0]]["X"])},
    "seeds": SEEDS, "common": COMMON, "conv": CONV,
    "data_root": str(DATA_ROOT), "data_sha256": data_hashes,
    "scenarios": SCENARIOS, "selection": "mean validation RMSE across scenarios",
    "variants": {
        "A0": "SAITS", "A1": "Conv-SAITS Full",
        "A2": "w/o Depth Conv, retain BatchNorm",
        "A3": "w/o Channel Mixing", "A4": "w/o Variable Mixing",
        "A5": "Single-scale K=15",
    },
})
save_json(RUN_DIR / "preprocessing_parameters.json", preprocessing)
print("Results:", RUN_DIR)
print("Device:", DEVICE)


## 3. Hàm chạy và lưu kết quả

In [ ]:
class Zero(nn.Module):
    def forward(self, x):
        return torch.zeros_like(x)


def build_model(variant, seed, output_dir):
    settings = {**COMMON, "seed": seed, "output_dir": output_dir}
    if variant == "A0":
        return SAITS(SAITSConfig(**settings))

    settings.update(CONV)
    if variant == "A5":
        settings["multi_scale_conv"] = False
    model = ConvSAITS(ConvSAITSConfig(**settings))
    encoder = model.backend.network.encoder
    if variant == "A2":
        encoder.temporal_branches = None
        encoder.temporal_fusion = None
        encoder.temporal = nn.Identity()
    elif variant == "A3":
        encoder.channel_ffn = Zero()
    elif variant == "A4":
        encoder.variable_ffn = Zero()
    return model


def evaluate_test(model, variant, seed, output_dir):
    rows = []
    for scenario in SCENARIOS:
        dataset = data["test"][scenario]
        prediction = model.impute({"X": dataset["X"]})
        if SAVE_PREDICTIONS:
            np.save(output_dir / f"prediction_{scenario.lower().replace('-', '_')}.npy", prediction)
        truth, mask = dataset["X_intact"], dataset["indicating_mask"]
        base = {"variant": variant, "seed": seed, "scenario": scenario}
        rows.append({
            **base, "log": "all", "units": "standardized",
            **compute_imputation_metrics(truth, prediction, mask, include_mape=False),
        })
        for index, log in enumerate(LOGS):
            target, estimate, scored = truth[..., index], prediction[..., index], mask[..., index]
            if not scored.any():
                continue 
            rows.append({
                **base, "log": log, "units": "standardized",
                **compute_imputation_metrics(target, estimate, scored, include_mape=False),
            })
            scale = preprocessing["normalization"][log]
            rows.append({
                **base, "log": log, "units": "original",
                **compute_imputation_metrics(
                    target * scale["std"] + scale["mean"],
                    estimate * scale["std"] + scale["mean"], scored,
                ),
            })
        pd.DataFrame(rows).to_csv(output_dir / "metrics.csv", index=False)


def run_variant(variant):
    if variant not in VARIANTS:
        return
    for seed in SEEDS:
        output_dir = RUN_DIR / variant / f"seed_{seed}"
        output_dir.mkdir(parents=True, exist_ok=True)
        set_seed(seed)
        model = build_model(variant, seed, output_dir)
        config = asdict(model.config)
        config["output_dir"] = str(output_dir)
        save_json(output_dir / "config.json", {"variant": variant, **config})
        for handler in logging.getLogger().handlers:
            handler.setFormatter(logging.Formatter(f"{variant} | seed={seed} | %(message)s"))
        started = synchronized_time(DEVICE)
        model.fit(train_data, validation_data)
        fit_seconds = synchronized_time(DEVICE) - started
        torch.save({
            "variant": variant, "config": config,
            "best_epoch": model.backend.best_epoch,
            "state_dict": model.backend.network.state_dict(),
        }, output_dir / "checkpoint.pt")
        pd.DataFrame(model.backend.training_history).to_csv(output_dir / "history.csv", index=False)
        save_json(output_dir / "run_info.json", {
            "parameters": sum(p.numel() for p in model.backend.network.parameters()),
            "fit_seconds": fit_seconds, "best_epoch": model.backend.best_epoch,
            "epochs_run": len(model.backend.training_history),
        })
        evaluate_test(model, variant, seed, output_dir)
        print(f"Done {variant}, seed={seed}: best epoch={model.backend.best_epoch}")
        del model
        gc.collect()
        torch.cuda.empty_cache()


## A0. SAITS

In [ ]:
run_variant("A0")

## A1. Conv-SAITS Full

In [ ]:
run_variant("A1")

## A2. w/o Depth Conv — giữ BatchNorm

In [ ]:
run_variant("A2")

## A3. w/o Channel Mixing

In [ ]:
run_variant("A3")

## A4. w/o Variable Mixing

In [ ]:
run_variant("A4")

## A5. Single-scale Conv-SAITS — K=15

In [ ]:
run_variant("A5")

## 4. Tổng hợp kết quả

In [ ]:
metrics = pd.concat(
    [pd.read_csv(RUN_DIR / variant / f"seed_{seed}" / "metrics.csv")
     for variant in VARIANTS for seed in SEEDS],
    ignore_index=True,
)
metrics.to_csv(RUN_DIR / "metrics_by_seed.csv", index=False)

metric_columns = ["rmse", "mae", "mse", "r2", "mape"]
average = metrics.groupby(["variant", "seed", "log", "units"], as_index=False)[metric_columns].mean()
average["scenario"] = "Average"
summary = pd.concat([metrics, average], ignore_index=True).groupby(
    ["variant", "scenario", "log", "units"], sort=False,
)[metric_columns].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary.reset_index().to_csv(RUN_DIR / "summary.csv", index=False)
print("Saved:", RUN_DIR / "summary.csv")


## 5. Đọc kết quả đã lưu

In [ ]:
VIEW_DIR = RUN_DIR  
protocol = json.loads((VIEW_DIR / "protocol.json").read_text(encoding="utf-8"))
SCENARIOS = protocol["scenarios"]
LOGS = json.loads((VIEW_DIR / "preprocessing_parameters.json").read_text(encoding="utf-8"))["log_columns"]
LABELS = {"A0": "SAITS", "A1": "Full", "A2": "w/o Depth Conv",
          "A3": "w/o Channel Mixing", "A4": "w/o Variable Mixing", "A5": "Single-scale"}
metrics = pd.read_csv(VIEW_DIR / "metrics_by_seed.csv")
summary = pd.read_csv(VIEW_DIR / "summary.csv")
order = [v for v in LABELS if v in metrics["variant"].unique()]